# Class 6. Parameterized Quantum Circuits and Variational Objectives (Exercises)

EVA: Quantum Machine Learning | ZHAW | Pavel Sulimov

---

Goals of this practice session:

1. Connect PQC architecture to representational capacity.
2. Derive expectation-value objectives analytically.
3. Build a compact VQE loop in Qiskit 2.x primitives.
4. Compare optimizers on the same variational task.
5. Cross-check one result in PennyLane.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from scipy.optimize import minimize

from qiskit import QuantumCircuit
from qiskit.primitives import StatevectorEstimator
from qiskit.quantum_info import SparsePauliOp, Statevector

import pennylane as qml

## Conceptual notes for this session

Qubits start in $\lvert 0\rangle$ by default. If a task needs a different starting basis state, apply gates such as $X$ (for $\lvert 1\rangle$) before the main encoding and ansatz blocks.

A common supervised PQC has the form
$$
\lvert\psi(x,\theta)\rangle = U_{\mathrm{ansatz}}(\theta)U_{\mathrm{enc}}(x)\lvert 0\rangle^{\otimes n}.
$$
The objective is typically an expectation value
$$
C(\theta)=\langle\psi(\theta)\rvert O\lvert\psi(\theta)\rangle.
$$
If $O=\sum_j c_jP_j$ is a Pauli sum, then
$$
C(\theta)=\sum_j c_j\langle P_j\rangle.
$$

Common audience questions:

- If quantum gates are linear, where does model nonlinearity come from? It comes from the nonlinear dependence of probabilities and losses on amplitudes, and from classical post-processing in hybrid loops.
- Why do we need multiple shots? Expectation values are estimated from samples on hardware, so the objective has finite-sample noise.
- Is a deeper ansatz always better? No. Greater depth can increase expressivity but can also worsen trainability and noise sensitivity.

---
## Part 1: Math tasks

### M6.1. Single-qubit ansatz and objective

For $\lvert\psi(\theta)\rangle=R_y(\theta)\lvert0\rangle$, derive $\langle Z\rangle$ and verify numerically on a grid.

In [ ]:
theta_grid = np.linspace(0.0, 2 * np.pi, 200)
z_expectation = ...  # YOUR CODE HERE

print("Closed form: <Z> = cos(theta)")
print("Sample values:")
for t in [0.0, np.pi / 2, np.pi]:
    print(f"theta={t:.3f}, <Z>={...:.6f}")  # YOUR CODE HERE

plt.figure(figsize=(6, 3))
plt.plot(theta_grid, z_expectation)
plt.xlabel(r"$\theta$")
plt.ylabel(r"$\langle Z \rangle$")
plt.tight_layout()
plt.show()

### M6.2. Toy VQE analytically

Let $H=Z$ and ansatz $R_y(\theta)\lvert0\rangle$. Find $\theta^*$ minimizing energy and confirm the minimum value.

In [ ]:
def energy_hz(theta: float) -> float:
    return ...  # YOUR CODE HERE

opt = minimize(lambda t: energy_hz(float(t[0])), x0=np.array([0.2]), method="COBYLA")

print(f"Numerical theta*: {float(opt.x[0]):.6f}")
print(f"Numerical E(theta*): {float(opt.fun):.6f}")
print("Analytical optimum: theta=pi (mod 2pi), E=-1")

---
## Part 2: Programming tasks

### P6.1. Qiskit estimator-based VQE objective

Build a two-qubit ansatz and evaluate
$$
H = Z\otimes Z + 0.5(X\otimes I + I\otimes X).
$$

In [ ]:
estimator = StatevectorEstimator()
hamiltonian = SparsePauliOp.from_list([
    ("ZZ", 1.0),
    ("XI", 0.5),
    ("IX", 0.5),
])

def ansatz_2q(params: np.ndarray) -> QuantumCircuit:
    qc = QuantumCircuit(2)
    qc.ry(float(params[0]), 0)
    qc.ry(float(params[1]), 1)
    qc.cx(0, 1)
    qc.ry(float(params[2]), 0)
    qc.ry(float(params[3]), 1)
    return qc

def energy_2q(params: np.ndarray) -> float:
    qc = ansatz_2q(params)
    ev = estimator.run([(qc, hamiltonian)]).result()[0].data.evs
    return ...  # YOUR CODE HERE

x_test = np.array([0.2, -0.1, 0.4, -0.3], dtype=float)
print(f"E(test) = {energy_2q(x_test):.6f}")

### P6.2. Optimize the two-qubit objective

Run at least two optimizers and compare convergence.

In [ ]:
x0 = np.array([0.3, -0.2, 0.5, -0.4], dtype=float)
methods = ["COBYLA", "Nelder-Mead", "Powell"]
histories = {}
final_values = {}

for method in methods:
    history = []

    def callback(xk):
        history.append(energy_2q(np.array(xk, dtype=float)))

    res = minimize(
        energy_2q,
        x0=x0,
        method=method,
        callback=callback,
        options={"maxiter": 80},
    )
    if not history:
        history.append(float(res.fun))
    histories[method] = history
    final_values[method] = float(res.fun)

for method, value in final_values.items():
    print(f"{method:12s} final energy: {value:.6f}")

plt.figure(figsize=(7, 3.5))
for method, history in histories.items():
    plt.plot(history, label=method)
plt.xlabel("iteration")
plt.ylabel("energy")
plt.legend()
plt.tight_layout()
plt.show()

### P6.3. Cost-landscape slice

Plot a one-parameter slice while freezing three parameters to see local structure.

In [ ]:
fixed = np.array([0.0, -0.2, 0.3, 0.1], dtype=float)
line = np.linspace(-np.pi, np.pi, 160)
vals = []

for t in line:
    p = fixed.copy()
    p[0] = t
    vals.append(energy_2q(p))

plt.figure(figsize=(6, 3))
plt.plot(line, vals)
plt.xlabel("parameter 0")
plt.ylabel("energy")
plt.tight_layout()
plt.show()

### P6.4. Short PennyLane cross-check

Compute one-qubit $\langle Z\rangle$ in PennyLane and compare with $\cos(\theta)$.

In [ ]:
dev = qml.device("default.qubit", wires=1)

@qml.qnode(dev)
def pl_expval(theta):
    qml.RY(theta, wires=0)
    return qml.expval(qml.PauliZ(0))

for theta in [0.1, 1.0, 2.2]:
    print(
        f"theta={theta:.2f} | PennyLane={pl_expval(theta):.6f} | "
        f"analytic={...:.6f}"
    )  # YOUR CODE HERE

---
## Summary

A variational model is a trainable family of states plus an expectation-value objective. The same physics appears across frameworks; practical behavior differs mostly due to ansatz design, optimizer choice, and noise in objective estimation.